## Adapter notebook: generate ratemaps and manifold dataset from initial (branch I) optimization results

In [1]:
import numpy as np
import pickle
import json
import os
import jax.numpy as jnp

from NRT_functions.helper_functions import init_irreps_2D

# ── Configuration ──────────────────────────────────────────────────────────────
filepath = './data/251106/170735'
counter  = 0          # which saved checkpoint

POSITIONS_PATH = os.path.abspath('../Pilot Decoder/data/manifold/positions.npy')

adding_string = '_' if counter != '' else ''

In [2]:
# ── Load optimization results ──────────────────────────────────────────────────
%cd "$filepath"

with open('parameters.json', 'r') as f:
    parameters = json.load(f)

with open(f'W{adding_string}{counter}.pkl', 'rb') as f:
    W = pickle.load(f)

with open(f'om{adding_string}{counter}.pkl', 'rb') as f:
    om = pickle.load(f)

# Prefer final weights if available
try:
    with open(f'W_final{adding_string}{counter}.pkl', 'rb') as f:
        W = pickle.load(f)
    print('Loaded W_final')
except FileNotFoundError:
    print('W_final not found, using W')

D = W.shape[0]
print(f'D={D}, W={W.shape}, om={om.shape}')

/home/julian/Projects/MasterThesis/ICLR_Actionable_Reps/data/251106/170735
Loaded W_final
D=64, W=(64, 63), om=(31, 2)


In [3]:
# ── Ratemaps ───────────────────────────────────────────────────────────────────
# Returns list of ratemaps, one per width; each entry has shape (D, res, res).

def get_ratemaps(W, om, res, widths):
    """Compute ratemaps on square grids of different room sizes.

    Args:
        W:      weight matrix, shape (D, 2*M+1)
        om:     frequencies, shape (M, 2)
        res:    grid resolution per side
        widths: iterable of absolute room widths (positions span [-w/2, w/2])

    Returns:
        List of arrays, each shape (D, res, res), one per width.
    """
    maps = []
    for w in widths:
        xs = np.linspace(-w / 2, w / 2, res)
        grid = np.meshgrid(xs, xs)                                        # each (res, res)
        phi = np.stack([grid[0].ravel(), grid[1].ravel()], axis=1)        # (res^2, 2)
        I = np.array(init_irreps_2D(om, phi))                             # (2*M+1, res^2)
        V = np.array(W) @ I                                               # (D, res^2)
        maps.append(V.reshape(D, res, res))
    return maps


res    = 70
widths = (1, 2, 4)

Vs = get_ratemaps(W, om, res, widths)
V_small, V_medium, V_large = Vs

print('V_small shape :', V_small.shape)   # spans [-0.5, 0.5]
print('V_medium shape:', V_medium.shape)  # spans [-1.0, 1.0]
print('V_large shape :', V_large.shape)   # spans [-2.0, 2.0]

# Save to disk
np.save(f'ratemaps_small_{counter}.npy',  V_small)
np.save(f'ratemaps_medium_{counter}.npy', V_medium)
np.save(f'ratemaps_large_{counter}.npy',  V_large)
print('Ratemaps saved.')

V_small shape : (64, 70, 70)
V_medium shape: (64, 70, 70)
V_large shape : (64, 70, 70)
Ratemaps saved.


In [4]:
# ── Manifold dataset ───────────────────────────────────────────────────────────
# Load absolute positions (B, L, 2), compute representation at each position,
# store as (B, L, D).

positions = np.load(POSITIONS_PATH)          # (B, L, 2)
B, L, _ = positions.shape
print(f'Positions: B={B}, L={L}')

# Flatten to (B*L, 2), compute representations, then reshape back
phi_flat = positions.reshape(B * L, 2)                          # (B*L, 2)
I_flat = np.array(init_irreps_2D(om, phi_flat))                 # (2*M+1, B*L)
G_flat = np.array(W) @ I_flat                                   # (D, B*L)
representations = G_flat.T.reshape(B, L, D)                     # (B, L, D)

print(f'Representations: {representations.shape}')  # (B, L, D)

np.save(f'representations_{counter}.npy', representations)
np.save(f'positions_manifold_{counter}.npy', positions)
print('Manifold dataset saved.')

Positions: B=100, L=1000
Representations: (100, 1000, 64)
Manifold dataset saved.
